# MWP Benchmark: itergen vs SDV

This notebook runs a compact benchmark on the healthcare sample in `demo/data` and compares:
- `itergen`
- `GaussianCopula` (SDV)
- `CTGAN` (SDV)

Metrics reported:
- quality score (SDMetrics `QualityReport`)
- correlation preservation (numeric correlation MAE)
- ML utility (TSTR accuracy + macro-F1)
- runtime (seconds)

In [7]:
from __future__ import annotations

import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from sdmetrics.reports.single_table import QualityReport
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer, GaussianCopulaSynthesizer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = Path.cwd()
if not (ROOT / "demo").exists() and (ROOT.parent / "demo").exists():
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from itergen import ItergenSynthesizer, RunConfig

warnings.filterwarnings("ignore", message=".*SingleTableMetadata.*")
warnings.filterwarnings("ignore", message=".*save_to_json.*")

SEED = 42
BENCHMARK_ROWS = 1200
CTGAN_EPOCHS = 20
ITERGEN_MAX_ATTEMPTS = 2
TARGET_COLUMN = "test_results"

DATA_PATH = ROOT / "demo" / "data" / "healthcare_dataset.csv"
CONFIG_PATH = ROOT / "demo" / "data" / "healthcare_extracted_config.yaml"
OUTPUT_DIR = ROOT / "demo" / "output" / "mwp_benchmark"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)

In [8]:
RENAME_MAP = {
    "Medical Condition": "medical_condition",
    "Gender": "gender",
    "Blood Type": "blood_type",
    "Insurance Provider": "insurance_provider",
    "Admission Type": "admission_type",
    "Medication": "medication",
    "Test Results": "test_results",
    "Age": "age",
    "Billing Amount": "billing_amount",
    "Room Number": "room_number",
}

KEEP_COLUMNS = [
    "medical_condition",
    "gender",
    "blood_type",
    "insurance_provider",
    "admission_type",
    "medication",
    "test_results",
    "age",
    "billing_amount",
    "room_number",
]

NUMERIC_COLUMNS = ["age", "billing_amount", "room_number"]

real_df = pd.read_csv(DATA_PATH).rename(columns=RENAME_MAP)
real_df = real_df[KEEP_COLUMNS].dropna()
if len(real_df) > BENCHMARK_ROWS:
    real_df = real_df.sample(BENCHMARK_ROWS, random_state=SEED)
real_df = real_df.reset_index(drop=True)

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=real_df)
metadata_dict = metadata.to_dict()

with CONFIG_PATH.open("r", encoding="utf-8") as fh:
    itergen_config = yaml.safe_load(fh)

real_df.shape

(1200, 10)

In [9]:
def evaluate_quality(real: pd.DataFrame, synth: pd.DataFrame) -> tuple[float, float]:
    report = QualityReport()
    report.generate(
        real_data=real,
        synthetic_data=synth,
        metadata=metadata_dict,
        verbose=False,
    )
    overall = float(report.get_score())
    props = report.get_properties()
    shapes = float(props.loc[props["Property"] == "Column Shapes", "Score"].iloc[0])
    return overall, shapes


def evaluate_correlation_mae(real: pd.DataFrame, synth: pd.DataFrame) -> float:
    real_corr = real[NUMERIC_COLUMNS].corr()
    synth_corr = synth[NUMERIC_COLUMNS].corr()
    diff = (real_corr - synth_corr).abs()
    mask = np.triu(np.ones(diff.shape), k=1).astype(bool)
    upper = diff.where(mask).stack()
    return float(upper.mean())


def evaluate_tstr(real: pd.DataFrame, synth: pd.DataFrame) -> tuple[float, float]:
    feature_cols = [c for c in real.columns if c != TARGET_COLUMN]
    cat_cols = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(real[c])]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    x_train = synth[feature_cols].copy()
    y_train = synth[TARGET_COLUMN].astype(str)
    x_test = real[feature_cols].copy()
    y_test = real[TARGET_COLUMN].astype(str)

    pre = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
            ("num", "passthrough", num_cols),
        ]
    )
    clf = Pipeline(
        steps=[
            ("pre", pre),
            (
                "rf",
                RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
            ),
        ]
    )
    clf.fit(x_train, y_train)
    pred = clf.predict(x_test)

    acc = float(accuracy_score(y_test, pred))
    f1 = float(f1_score(y_test, pred, average="macro"))
    return acc, f1


def benchmark_row(
    name: str, synthetic_df: pd.DataFrame, runtime_sec: float
) -> dict[str, float | str]:
    synth = synthetic_df[KEEP_COLUMNS].dropna().reset_index(drop=True)

    q_overall, q_shapes = evaluate_quality(real_df, synth)
    corr_mae = evaluate_correlation_mae(real_df, synth)
    tstr_acc, tstr_f1 = evaluate_tstr(real_df, synth)

    return {
        "generator": name,
        "quality_overall": q_overall,
        "quality_column_shapes": q_shapes,
        "correlation_mae": corr_mae,
        "tstr_accuracy": tstr_acc,
        "tstr_macro_f1": tstr_f1,
        "runtime_sec": runtime_sec,
    }

In [10]:
results = []

start = time.perf_counter()
itergen_result = ItergenSynthesizer(
    itergen_config,
    RunConfig(
        n_rows=len(real_df),
        seed=SEED,
        max_attempts=ITERGEN_MAX_ATTEMPTS,
        save_output=False,
        log_level="quiet",
    ),
).generate()
itergen_runtime = time.perf_counter() - start
results.append(benchmark_row("itergen", itergen_result.dataframe, itergen_runtime))

start = time.perf_counter()
gc = GaussianCopulaSynthesizer(metadata=metadata)
gc.fit(real_df)
synthetic_gc = gc.sample(num_rows=len(real_df))
gc_runtime = time.perf_counter() - start
results.append(benchmark_row("gaussian_copula", synthetic_gc, gc_runtime))

start = time.perf_counter()
ctgan = CTGANSynthesizer(metadata=metadata, epochs=CTGAN_EPOCHS, verbose=False)
ctgan.fit(real_df)
synthetic_ctgan = ctgan.sample(num_rows=len(real_df))
ctgan_runtime = time.perf_counter() - start
results.append(benchmark_row(f"ctgan_e{CTGAN_EPOCHS}", synthetic_ctgan, ctgan_runtime))

results_df = pd.DataFrame(results)
results_df

,generator,quality_overall,quality_column_shapes,correlation_mae,tstr_accuracy,tstr_macro_f1,runtime_sec
0,itergen,0.951333,0.951333,0.017816,0.324167,0.322213,108.321621
1,gaussian_copula,0.974833,0.974833,0.008992,0.358333,0.348311,0.196962
2,ctgan_e20,0.900250,0.900250,0.017678,0.352500,0.329111,3.857832


In [11]:
summary_df = results_df.sort_values("quality_overall", ascending=False).reset_index(
    drop=True
)
summary_df

,generator,quality_overall,quality_column_shapes,correlation_mae,tstr_accuracy,tstr_macro_f1,runtime_sec
0,gaussian_copula,0.974833,0.974833,0.008992,0.358333,0.348311,0.196962
1,itergen,0.951333,0.951333,0.017816,0.324167,0.322213,108.321621
2,ctgan_e20,0.900250,0.900250,0.017678,0.352500,0.329111,3.857832


In [12]:
output_csv = OUTPUT_DIR / "metrics_comparison.csv"
summary_df.to_csv(output_csv, index=False)
output_csv

PosixPath('/Users/temurmamanazarov/Work/Swansea-Uni/Code/itergen/demo/output/mwp_benchmark/metrics_comparison.csv')